In [ ]:
from uuid import UUID

from core.processors.challenge import ChallengeProcessor
from db.conf import create_db_engine, get_async_session
from dotenv import find_dotenv, load_dotenv

from db.repositories.jobs import JobRepository, CHALLENGE_GEN_JOB_NAME

load_dotenv(find_dotenv())
engine = create_db_engine()
db = get_async_session(engine)

In [ ]:
from core.agents.common import TemplateManager, gpt_5_nano, medium_effort_gpt_5
from core.agents.challenge import ChallengeGenAgent, ChallengeLoader

loader = ChallengeLoader(db)
agent = ChallengeGenAgent(gpt_5_nano(), medium_effort_gpt_5(), loader, TemplateManager())
challenge_processor = ChallengeProcessor(agent, db)

In [ ]:
from db.repositories.jobs import ChallengeGenJob

async with db() as session:
  job_repo = JobRepository(session)
  job = await job_repo.create_job(CHALLENGE_GEN_JOB_NAME,
                                  ChallengeGenJob(
                                      video_id=UUID("738ee508-b0c7-11f0-8c56-33d44c2a65c3"),
                                      pattern_group_id=UUID("249b2e88-b302-11f0-bc33-7f2eac94b24a"))
                                  )
  await session.commit()

job

In [ ]:
result = await challenge_processor.run(job.id)
result